In [1]:
# =========================================================
# STEP 1 — INSTALL LIBRARIES
# =========================================================

!pip install -q librosa soundfile

# =========================================================
# STEP 2 — MOUNT GOOGLE DRIVE
# =========================================================

from google.colab import drive
drive.mount('/content/drive')

# =========================================================
# STEP 3 — IMPORTS
# =========================================================

import os
import numpy as np
import librosa
import tensorflow as tf

from sklearn.model_selection import train_test_split
from sklearn.utils.class_weight import compute_class_weight

from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import (
    Conv1D,
    MaxPooling1D,
    BatchNormalization,
    Dropout,
    Bidirectional,
    LSTM,
    Dense
)

from tensorflow.keras.callbacks import (
    EarlyStopping,
    ReduceLROnPlateau,
    ModelCheckpoint
)

# =========================================================
# STEP 4 — CONFIGURATION
# =========================================================

BASE_PATH = "/content/drive/MyDrive/Binary_Model/binary_model"

NOT_CRY_PATH = os.path.join(BASE_PATH, "not_cry")
CRY_PATH = os.path.join(BASE_PATH, "cry")


MODEL_SAVE_PATH = os.path.join(
    BASE_PATH,
    "best_binary_cry_model.keras"
)

SAMPLE_RATE = 16000
N_MFCC = 40
MAX_LEN = 100

# =========================================================
# STEP 5 — FEATURE EXTRACTION
# =========================================================

def extract_features(file_path):

    try:

        audio, sr = librosa.load(
            file_path,
            sr=SAMPLE_RATE
        )

        # Normalize
        audio = librosa.util.normalize(audio)

        # MFCC
        mfcc = librosa.feature.mfcc(
            y=audio,
            sr=sr,
            n_mfcc=N_MFCC
        )

        # Delta
        delta = librosa.feature.delta(mfcc)

        # Delta Delta
        delta2 = librosa.feature.delta(
            mfcc,
            order=2
        )

        # Combine
        features = np.concatenate(
            [mfcc, delta, delta2],
            axis=0
        )

        # Padding
        if features.shape[1] < MAX_LEN:

            pad_width = MAX_LEN - features.shape[1]

            features = np.pad(
                features,
                pad_width=((0,0),(0,pad_width)),
                mode='constant'
            )

        else:

            features = features[:, :MAX_LEN]

        return features.T

    except Exception as e:

        print(f"Error processing {file_path}")
        print(e)

        return None

# =========================================================
# STEP 6 — LOAD DATASET
# =========================================================

X = []
y = []

# Cry
for file_name in os.listdir(CRY_PATH):

    file_path = os.path.join(
        CRY_PATH,
        file_name
    )

    features = extract_features(file_path)

    if features is not None:

        X.append(features)
        y.append(1)

# Not Cry
for file_name in os.listdir(NOT_CRY_PATH):

    file_path = os.path.join(
        NOT_CRY_PATH,
        file_name
    )

    features = extract_features(file_path)

    if features is not None:

        X.append(features)
        y.append(0)

# Convert
X = np.array(X)
y = np.array(y)

print("Feature Shape:", X.shape)
print("Labels Shape:", y.shape)

# =========================================================
# STEP 7 — TRAIN TEST SPLIT
# =========================================================

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

print("X_train:", X_train.shape)
print("X_test:", X_test.shape)

# =========================================================
# STEP 8 — NORMALIZATION
# =========================================================

mean = np.mean(X_train)
std = np.std(X_train)

X_train = (X_train - mean) / std
X_test = (X_test - mean) / std

# Save normalization values
np.save(
    os.path.join(BASE_PATH, "mean.npy"),
    mean
)

np.save(
    os.path.join(BASE_PATH, "std.npy"),
    std
)

# =========================================================
# STEP 9 — CLASS WEIGHTS
# =========================================================

class_weights = compute_class_weight(
    class_weight='balanced',
    classes=np.unique(y_train),
    y=y_train
)

class_weights = {
    0: class_weights[0],
    1: class_weights[1]
}

print(class_weights)

# =========================================================
# STEP 10 — BUILD MODEL
# =========================================================

model = Sequential([

    Conv1D(
        64,
        3,
        activation='relu',
        input_shape=(MAX_LEN, 120)
    ),

    BatchNormalization(),

    MaxPooling1D(2),

    Dropout(0.3),

    Conv1D(
        128,
        3,
        activation='relu'
    ),

    BatchNormalization(),

    MaxPooling1D(2),

    Dropout(0.3),

    Bidirectional(
        LSTM(
            64,
            return_sequences=False
        )
    ),

    Dropout(0.4),

    Dense(
        64,
        activation='relu'
    ),

    Dropout(0.3),

    Dense(
        1,
        activation='sigmoid'
    )
])

# =========================================================
# STEP 11 — COMPILE MODEL
# =========================================================

model.compile(
    optimizer='adam',
    loss='binary_crossentropy',
    metrics=['accuracy']
)

model.summary()

# =========================================================
# STEP 12 — CALLBACKS
# =========================================================

early_stop = EarlyStopping(
    monitor='val_loss',
    patience=5,
    restore_best_weights=True
)

reduce_lr = ReduceLROnPlateau(
    monitor='val_loss',
    factor=0.5,
    patience=2,
    verbose=1
)

checkpoint = ModelCheckpoint(
    MODEL_SAVE_PATH,
    monitor='val_accuracy',
    save_best_only=True,
    verbose=1
)

# =========================================================
# STEP 13 — TRAIN MODEL
# =========================================================

history = model.fit(
    X_train,
    y_train,
    epochs=30,
    batch_size=8,
    validation_data=(X_test, y_test),
    class_weight=class_weights,
    callbacks=[
        early_stop,
        reduce_lr,
        checkpoint
    ]
)

# =========================================================
# STEP 14 — EVALUATION
# =========================================================

loss, accuracy = model.evaluate(
    X_test,
    y_test
)

print("\\nTest Accuracy:", accuracy)

Mounted at /content/drive
Feature Shape: (87, 100, 120)
Labels Shape: (87,)
X_train: (69, 100, 120)
X_test: (18, 100, 120)
{0: np.float64(0.9857142857142858), 1: np.float64(1.0147058823529411)}


/usr/local/lib/python3.12/dist-packages/keras/src/layers/convolutional/base_conv.py:113: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ conv1d (Conv1D)                 │ (None, 98, 64)         │        23,104 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization             │ (None, 98, 64)         │           256 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling1d (MaxPooling1D)    │ (None, 49, 64)         │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 49, 64)         │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv1d_1 (Conv1D)               │ (None, 47, 128)        │        24,704 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_1           │ (None, 47, 128)        │           512 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling1d_1 (MaxPooling1D)  │ (None, 23, 128)        │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_1 (Dropout)             │ (None, 23, 128)        │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ bidirectional (Bidirectional)   │ (None, 128)            │        98,816 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_2 (Dropout)             │ (None, 128)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 64)             │         8,256 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_3 (Dropout)             │ (None, 64)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 1)              │            65 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 155,713 (608.25 KB)

 Trainable params: 155,329 (606.75 KB)

 Non-trainable params: 384 (1.50 KB)

Epoch 1/30
9/9 ━━━━━━━━━━━━━━━━━━━━ 0s 23ms/step - accuracy: 0.5418 - loss: 0.6903
Epoch 1: val_accuracy improved from None to 0.50000, saving model to /content/drive/MyDrive/Binary_Model/binary_model/best_binary_cry_model.keras

Epoch 1: finished saving model to /content/drive/MyDrive/Binary_Model/binary_model/best_binary_cry_model.keras
9/9 ━━━━━━━━━━━━━━━━━━━━ 9s 180ms/step - accuracy: 0.6232 - loss: 0.6522 - val_accuracy: 0.5000 - val_loss: 0.8883 - learning_rate: 0.0010
Epoch 2/30
6/9 ━━━━━━━━━━━━━━━━━━━━ 0s 12ms/step - accuracy: 0.7111 - loss: 0.5658
Epoch 2: val_accuracy did not improve from 0.50000
9/9 ━━━━━━━━━━━━━━━━━━━━ 0s 22ms/step - accuracy: 0.7391 - loss: 0.5470 - val_accuracy: 0.4444 - val_loss: 0.7684 - learning_rate: 0.0010
Epoch 3/30
6/9 ━━━━━━━━━━━━━━━━━━━━ 0s 11ms/step - accuracy: 0.7868 - loss: 0.4897
Epoch 3: val_accuracy did not improve from 0.50000
9/9 ━━━━━━━━━━━━━━━━━━━━ 0s 21ms/step - accuracy: 0.7826 - loss: 0.4912 - val_accuracy: 0.4444 - val_loss: 0.7184 

In [13]:
# =========================================================
# STEP 1 — IMPORTS
# =========================================================

import os
import numpy as np
import librosa
import tensorflow as tf

# =========================================================
# STEP 2 — CONFIGURATION
# =========================================================

BASE_PATH = "/content/drive/MyDrive/Binary_Model/binary_model"

MODEL_PATH = os.path.join(
    BASE_PATH,
    "best_binary_cry_model.keras"
)

MEAN_PATH = os.path.join(
    BASE_PATH,
    "mean.npy"
)

STD_PATH = os.path.join(
    BASE_PATH,
    "std.npy"
)

TEST_AUDIO ="/content/drive/MyDrive/Binary_Model/binary_model/test_data/audio3_fan_tv.wav"

SAMPLE_RATE = 16000
N_MFCC = 40
MAX_LEN = 100

# =========================================================
# STEP 3 — LOAD MODEL
# =========================================================

model = tf.keras.models.load_model(
    MODEL_PATH
)

mean = np.load(MEAN_PATH)
std = np.load(STD_PATH)

print("Model Loaded Successfully")

# =========================================================
# STEP 4 — FEATURE EXTRACTION
# =========================================================

def extract_features(file_path):

    audio, sr = librosa.load(
        file_path,
        sr=SAMPLE_RATE
    )

    audio = librosa.util.normalize(audio)

    mfcc = librosa.feature.mfcc(
        y=audio,
        sr=sr,
        n_mfcc=N_MFCC
    )

    delta = librosa.feature.delta(mfcc)

    delta2 = librosa.feature.delta(
        mfcc,
        order=2
    )

    features = np.concatenate(
        [mfcc, delta, delta2],
        axis=0
    )

    if features.shape[1] < MAX_LEN:

        pad_width = MAX_LEN - features.shape[1]

        features = np.pad(
            features,
            pad_width=((0,0),(0,pad_width)),
            mode='constant'
        )

    else:

        features = features[:, :MAX_LEN]

    return features.T

# =========================================================
# STEP 5 — PROCESS AUDIO
# =========================================================

features = extract_features(TEST_AUDIO)

# Normalize
features = (features - mean) / std

# Expand dimensions
features = np.expand_dims(
    features,
    axis=0
)

# =========================================================
# STEP 6 — PREDICT
# =========================================================

prediction = model.predict(features)[0][0]

# =========================================================
# STEP 7 — FINAL RESULT
# =========================================================

if prediction > 0.5:

    print("\nPrediction: CRY")

else:

    print("\nPrediction: NOT CRY")

print("\nConfidence Score:", float(prediction))

Model Loaded Successfully


1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 368ms/step

Prediction: NOT CRY

Confidence Score: 0.02974628657102585
